# 10 -- LLM + MCTS (plan v5 Step 17): timing probe, control, then self-play training

Step 17 asks: from the same SFT start (the 500k run's step 30k, end of pass 1), does **self-play with search**
go further than **more imitation** (SFT pass 2) for the same extra GPU-hours?

This notebook runs it in five parts:
1. **Timing probe** (~5 min): how fast the LLM drives a batched search on this GPU -- one search move across 64
   games at 8 and 16 simulations. Turns the plan's throughput estimate into a measurement.
2. **Control** (~2 h at 16 simulations, ~1 h at 8): the start checkpoint rated **with search at play time**, no
   extra training, against the same opponents as its no-search rating. Its Elo gain over no-search is the go /
   no-go check for part 3: a clear gain means the search's move choices beat the policy's, so they are worth
   training on.
3. **Gate:** refuses to train unless the control's search gain is +50 Elo or more (Derek, Nov 1).
4. **Train the self-play LLM** (`train/llm_mcts.py`, `configs/llm_mcts.yaml`) to 10.0 GPU-h; `SMOKE=1` = 2 tiny
   rounds in a separate `_smoke` folder (checks memory, speed, saving) instead.
5. **Evals, the 2 x 2** (plan Step 17): the self-play LLM without and with search, and SFT @62.5k with search, all
   on the full panel (Step 12f; waits until its rungs are frozen). SFT @62.5k without search is the rung itself.

Switches come from the shell: `PROBE=1`, `CONTROL_SIMS=16` (or `16,8`; empty = skip), `TRAIN=1`, `SMOKE=1`,
`EVAL=1`. Training and evals resume where they stopped. Every control game goes to
`results/games.csv`; a rerun skips a control rating that already has its full games. Needs a GPU pod.

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
import os
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import json
import yaml
import torch
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
print("GPU:", GPU)

## Settings
The start checkpoint (Step 17's phase 1) and the control's opponents: the same as the 500k checkpoints'
no-search ratings (runbook I), so the two rows compare directly.

In [ ]:
PROBE = os.environ.get("PROBE", "1") == "1"
CONTROL_SIMS = [int(x) for x in os.environ.get("CONTROL_SIMS", "16").split(",") if x.strip()]
TRAIN = os.environ.get("TRAIN", "0") == "1"      # part 4: train the self-play LLM
SMOKE = os.environ.get("SMOKE", "0") == "1"      # with TRAIN: 2 tiny rounds in a separate _smoke folder
EVAL = os.environ.get("EVAL", "0") == "1"        # part 5: the 2 x 2 on the full panel
GATE_ELO = 50                                    # search must add at least this at play time (Derek, Nov 1)
SEARCH_SIMS = 16
START_RUN, START_STEP = "structured_masked_nall_e2_s0_scaled_d12_n500000", 30000
START = {"name": f"{START_RUN}@{START_STEP}", "kind": "sft", "run": START_RUN, "step": START_STEP}
OPPONENTS = ["greedy", "step_20000", "sf_1320"]
DATA_DIR = os.path.join(ROOT, "data", "processed", "sft_scaled_d12_n500000")
print("start:", START["name"], "| probe:", PROBE, "| control simulations:", CONTROL_SIMS, "| opponents:", OPPONENTS)
print("train:", TRAIN, "| smoke:", SMOKE, "| eval:", EVAL)

## 1. Timing probe
Batched evaluation speed (positions per second at 32 / 64 / 128 positions per call), then one search move across
64 games at 8 and 16 simulations. The projection uses the plan's estimate for the training update (~0.2 s per
position); part 3's smoke test measures that too.

In [ ]:
import random
import chess
import time
from eval.agents import checkpoint_dir
from eval.match import random_opening
from model.llm_evaluator import LLMEvaluator, time_search
from train.sft import load_checkpoint

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

if PROBE:
    policy = load_checkpoint(checkpoint_dir(START, ROOT), name=START["name"])
    ev = LLMEvaluator(policy, batch_positions=128)
    rng = random.Random(0)
    boards = []
    for _ in range(128):
        b = chess.Board()
        for m in random_opening(rng, 12):
            b.push_uci(m)
        boards.append(b)
    ev.evaluate(boards[:8])                                    # warm-up (CUDA kernels, caches)
    for n in (32, 64, 128):
        sync(); t0 = time.perf_counter()
        ev.evaluate(boards[:n])
        sync(); dt = time.perf_counter() - t0
        print(f"evaluate {n:>3} positions: {dt:.2f}s -> {1000 * dt / n:.1f} ms/position")
    probe = {}
    for sims in (8, 16):
        r = time_search(ev, boards[:64], sims)
        probe[sims] = r
        per_position = r["seconds_per_game_move"] + 0.2           # + the update's estimate
        n_pos = 9.6 * 3600 / per_position
        print(f"{sims} simulations: one move of 64 games {r['seconds']:.1f}s -> {r['seconds_per_game_move']:.3f}s per "
              f"game-move ({1000 * r['seconds_per_position']:.1f} ms/position, {r['model_calls']} model calls); "
              f"~{n_pos / 1000:.0f}k training positions (~{n_pos / 100:.0f} games) in 9.6 GPU-h")
    if torch.cuda.is_available():
        print("peak GPU memory:", round(torch.cuda.max_memory_allocated() / 1e9, 2), "GB")
    del policy, ev
    torch.cuda.empty_cache()
else:
    print("PROBE is off")

## 2. Control: the start checkpoint with search at play time
Rows `<run>_mcts<N>@<step>` (their own series), carrying the start checkpoint's training GPU-hours: search adds
no training compute, its cost shows in sec/move. ~1.5 s/move at 16 simulations.

In [ ]:
from eval.agents import agent_name, is_rerated, rate
from eval.anchors import ResultsTable, build_reference_agents, close_agents
from eval.elo import format_table
from train.sft import gpu_hours_at

eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
GAMES, BATCH = eval_cfg["games_per_opponent"], eval_cfg["batch_games"]
results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
todo = [s for s in CONTROL_SIMS if not is_rerated(results, agent_name(START["name"], s), GAMES * len(OPPONENTS))]
for s in CONTROL_SIMS:
    if s not in todo:
        print(f"{agent_name(START['name'], s)}: already rated with its full games -- skipped")
if todo:
    run_dir = os.path.join(ROOT, "checkpoints", "sft", START_RUN)
    label_cpu = json.load(open(os.path.join(DATA_DIR, "manifest.json")))["stockfish_cpu_seconds"] / 3600
    opponents = build_reference_agents(results, OPPONENTS, ROOT)
    try:
        for sims in todo:
            name = agent_name(START["name"], sims)
            print(f"\n== {name} vs {OPPONENTS}")
            est = rate({**START, "name": name, "simulations": sims}, results, ROOT, opponents,
                       fields={"run": f"{START_RUN}_mcts{sims}", "step": START_STEP, "seed": 0,
                               "train_gpu_hours": round(gpu_hours_at(run_dir, START_STEP), 4),
                               "label_cpu_hours": round(label_cpu, 4), "dataset_size": 500000,
                               "note": f"Step 17 control: 500k SFT step {START_STEP} + MCTS {sims} sims at play, no extra training"},
                       games_per_opponent=GAMES, batch_games=BATCH, gpu=GPU)
            print(format_table([est]))
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)

## Search gain
The start checkpoint without search (from runbook I's evals, if done) next to each control row. The gain is the
go / no-go number for part 3.

In [ ]:
import pandas as pd

t = ResultsTable(results.path)
names = [START["name"]] + [agent_name(START["name"], s) for s in CONTROL_SIMS]
rows = pd.DataFrame([t.rows[n] for n in names if n in t.rows])
display(rows[["name", "rating", "ci_low", "ci_high", "games", "wins", "draws", "losses", "sec_per_move"]])
if START["name"] in t.rows:
    base = float(t.rows[START["name"]]["rating"])
    for s in CONTROL_SIMS:
        n = agent_name(START["name"], s)
        if n in t.rows:
            print(f"{s} simulations add {float(t.rows[n]['rating']) - base:+.0f} Elo over no search")
else:
    print("no no-search row for the start checkpoint yet -- rate it first (runbook I)")

## 3. Gate: search must add +50 Elo
Reads the control row and the start's no-search row from `results.csv`. Training is refused (the notebook stops
with an error, so a chained command stops too) unless the control has all its games and adds at least
`GATE_ELO`.

In [ ]:
from eval.agents import agent_name
from eval.anchors import ResultsTable

if TRAIN and not SMOKE:
    t = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
    ctrl = t.rows.get(agent_name(START["name"], SEARCH_SIMS))
    base = t.rows.get(START["name"])
    if not ctrl or not base or int(ctrl["games"] or 0) < GAMES * len(OPPONENTS):
        raise RuntimeError("the control isn't fully rated yet -- run part 2 first")
    gain = float(ctrl["rating"]) - float(base["rating"])
    print(f"control {float(ctrl['rating']):.0f} vs no search {float(base['rating']):.0f}: {gain:+.0f} Elo (gate: +{GATE_ELO})")
    if gain < GATE_ELO:
        raise RuntimeError(f"search adds {gain:+.0f} < +{GATE_ELO} Elo: no-go, training not started")
    print("GO")
else:
    print("gate checked only for the real training run")

## 4. Train the self-play LLM
Settings in `configs/llm_mcts.yaml` (agreed Nov 1): 64 games per round, 16 simulations, no pull toward the start,
the last 3 rounds' positions, each new position sampled ~2x, SFT's learning rate and batch size; stops at 10.0
GPU-h (within about half a round). Resumable: rerun to continue. The log (`mcts_log.csv` in the run folder) has one
row per round.

In [ ]:
import dataclasses
from train.llm_mcts import MCTSTrainConfig, run_name, start_policy, train

CFG = MCTSTrainConfig.from_dict(yaml.safe_load(open("configs/llm_mcts.yaml")))
if SMOKE:
    CFG = dataclasses.replace(CFG, games_per_round=8, max_plies=40, rounds=2, checkpoint_every=1, tag="smoke")
RUN = run_name(CFG)
RUN_DIR = os.path.join(ROOT, "checkpoints", "llm_mcts", RUN)
print("run:", RUN)
if TRAIN:
    policy, init_h = start_policy(CFG, ROOT)
    print(f"start: {CFG.init_run}@{CFG.init_step} ({init_h:.2f} GPU-h) | budget {CFG.budget_gpu_hours} GPU-h")
    summary = train(policy, CFG, RUN_DIR, init_gpu_hours=init_h, extra_info={"gpu": GPU})
    print(json.dumps(summary, indent=2))
    if SMOKE:
        import csv
        last = list(csv.DictReader(open(os.path.join(RUN_DIR, "mcts_log.csv"))))[-1]
        per_pos = (float(last["selfplay_seconds"]) + float(last["update_seconds"])) / max(int(last["positions"]), 1)
        print(f"smoke: {per_pos:.2f} s per position (short games; full games batch better) "
              f"-> ~{CFG.budget_gpu_hours * 3600 / per_pos:,.0f} positions in the full budget")
    del policy
    torch.cuda.empty_cache()
else:
    print("TRAIN off")

## 5. Evals: the 2 x 2 on the full panel
| | no search | with search |
|---|---|---|
| SFT @62.5k | the rung's own rating (Step 12f) | rated here |
| self-play LLM (final checkpoint) | rated here | rated here |

Each new row plays every panel member 20 times (120 games), saved after every batch; a rerun continues. Waits (up
to 8 h, checking every 10 min) until Step 12(f) has frozen the panel's rungs.

In [ ]:
import time as _time
from eval.anchors import build_reference_agents, close_agents
from eval.panel import extend, load_panel
from train.llm_mcts import gpu_hours_at as mcts_gpu_hours_at, list_checkpoints as mcts_checkpoints
from train.sft import gpu_hours_at as sft_gpu_hours_at

if EVAL and not SMOKE:
    PANEL = load_panel()
    deadline = _time.time() + 8 * 3600
    while not all(ResultsTable(os.path.join(ROOT, eval_cfg["results_file"])).role(n) in ("pin", "reference") for n in PANEL):
        if _time.time() > deadline:
            raise RuntimeError("the panel's rungs still aren't frozen -- run notebook 11 stage 1")
        print("waiting for the panel's rungs (notebook 11, stage 1)...")
        _time.sleep(600)
    results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
    label_cpu = json.load(open(os.path.join(DATA_DIR, "manifest.json")))["stockfish_cpu_seconds"] / 3600
    final = mcts_checkpoints(RUN_DIR)[-1][0]
    sp = f"{RUN}@r{final}"
    sp_fields = {"step": final, "seed": 0, "train_gpu_hours": round(mcts_gpu_hours_at(RUN_DIR, final), 4),
                 "label_cpu_hours": label_cpu, "dataset_size": 500000}
    sft = f"{START_RUN}@62500"
    sft_dir = os.path.join(ROOT, "checkpoints", "sft", START_RUN)
    SPECS = [   # (load spec, row fields)
        ({"name": sp, "kind": "llm_mcts", "run": RUN, "step": final},
         {**sp_fields, "run": f"llm_mcts/{RUN}", "note": f"self-play LLM: 16-sim search self-play from {START['name']}"}),
        ({"name": agent_name(sp, SEARCH_SIMS), "kind": "llm_mcts", "run": RUN, "step": final, "simulations": SEARCH_SIMS},
         {**sp_fields, "run": f"llm_mcts/{RUN}_mcts{SEARCH_SIMS}", "note": "self-play LLM, 16-sim search at play"}),
        ({"name": agent_name(sft, SEARCH_SIMS), "kind": "sft", "run": START_RUN, "step": 62500, "simulations": SEARCH_SIMS},
         {"run": f"{START_RUN}_mcts{SEARCH_SIMS}", "step": 62500, "seed": 0, "label_cpu_hours": label_cpu,
          "dataset_size": 500000, "train_gpu_hours": round(sft_gpu_hours_at(sft_dir, 62500), 4),
          "note": "SFT pass 2 end, 16-sim search at play"}),
    ]
    opponents = build_reference_agents(results, PANEL, ROOT)
    try:
        for spec, fields in SPECS:
            print(f"\n== {spec['name']}")
            extend(spec, results, ROOT, opponents, games_per_opponent=GAMES, batch_games=BATCH, fields=fields, gpu=GPU)
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)
    t = ResultsTable(results.path)
    cells = {("SFT @62.5k", "no search"): sft, ("SFT @62.5k", "with search"): agent_name(sft, SEARCH_SIMS),
             ("self-play LLM", "no search"): sp, ("self-play LLM", "with search"): agent_name(sp, SEARCH_SIMS)}
    for (model, mode), name in cells.items():
        r = t.rows.get(name, {})
        print(f"{model:<14} {mode:<12} {r.get('rating', '-'):>7} [{r.get('ci_low', '')}, {r.get('ci_high', '')}]  "
              f"W/D/L {r.get('wins', '')}/{r.get('draws', '')}/{r.get('losses', '')}  GPU-h {r.get('train_gpu_hours', '')}")
else:
    print("EVAL off")